In [1]:
print("hello")

hello


In [2]:
# %pip install -q pypdf langchain-text-splitters
%pip install pypdf langchain-text-splitters langchain-community

Note: you may need to restart the kernel to use updated packages.


# chunking

In [3]:
from pathlib import Path
import re

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


SECTION_HEADINGS = [
    "What .* is and what it is used for",
    "Before you take",
    "How to take",
    "Possible side effects",
    "Use in pregnancy and breast-feeding",
    "How to store",
]

HEADING_PATTERN = re.compile(
    r"^(?:\d+\.\s*)?(" + "|".join(SECTION_HEADINGS) + r").*$",
    re.IGNORECASE
)


def split_by_section(text):
    sections = []
    current_section = "Overview"
    current_text = []

    for line in text.splitlines():
        line = line.strip()

        if HEADING_PATTERN.match(line):
            if current_text:
                sections.append((current_section, "\n".join(current_text).strip()))

            current_section = line
            current_text = []
        else:
            current_text.append(line)

    if current_text:
        sections.append((current_section, "\n".join(current_text).strip()))

    return sections


def get_medicine_name(text):
    first_lines = text.splitlines()

    for line in first_lines:
        if line.strip():
            return line.strip()

    return "Unknown"


def chunk_documents(docs_path="docs", chunk_size=1000, chunk_overlap=100):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    all_chunks = []

    for pdf_path in Path(docs_path).glob("*.pdf"):
        print(f"Processing: {pdf_path.name}")

        pages = PyPDFLoader(str(pdf_path)).load()
        text = "\n".join(page.page_content for page in pages)

        medicine_name = get_medicine_name(text)

        sections = split_by_section(text)

        for section, content in sections:
            chunks = splitter.split_text(content)

            for i, chunk in enumerate(chunks):
                chunk_text = f"MEDICINE: {medicine_name}\nSECTION: {section}\n\n{chunk}"

                all_chunks.append({
                    "id": f"{pdf_path.name}::{section}::{i}",
                    "source": pdf_path.name,
                    "medicine": medicine_name,
                    "section": section,
                    "text": chunk_text
                })

    return all_chunks

/var/folders/fd/tjdwkh6x2tl4_t9qrrkhy_2h0000gn/T/ipykernel_65862/3401177121.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/opt/miniconda3/envs/experiment/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Chunking call

In [4]:

docs_path="docs"
chunk_size=1000
chunk_overlap=100

chunks = chunk_documents(docs_path,chunk_size,chunk_overlap)
print(chunks)


print(f"Total chunks: {len(chunks)}")

for chunk in chunks[:35]:
    print("=" * 80)
    print("SOURCE:", chunk["source"])
    print("SECTION:", chunk["section"])
    print("TEXT:")
    print(chunk["text"])
    
    
# Save into a txt file for loggings     
with open("chunks.txt", "w", encoding="utf-8") as f:
    for i, chunk in enumerate(chunks, 1):
        f.write("=" * 100 + "\n")
        f.write(f"CHUNK {i}\n")
        f.write("=" * 100 + "\n")
        f.write(f"ID: {chunk['id']}\n")
        f.write(f"SOURCE: {chunk['source']}\n")
        f.write(f"SECTION: {chunk['section']}\n")
        f.write("-" * 100 + "\n")
        f.write(chunk["text"])
        f.write("\n\n")

print("Saved chunks to chunks.txt")

Processing: maxpro_20mg_esomeprazole_leaflet.pdf
Processing: fenadin_120mg_fexofenadine_leaflet.pdf
Processing: rolip_10mg_rosuvastatin_leaflet.pdf
Processing: doxicap_100mg_doxycycline_leaflet.pdf
Processing: rolac_10mg_ketorolac_leaflet.pdf
[{'id': 'maxpro_20mg_esomeprazole_leaflet.pdf::Overview::0', 'source': 'maxpro_20mg_esomeprazole_leaflet.pdf', 'medicine': 'Maxpro 20 mg', 'section': 'Overview', 'text': 'MEDICINE: Maxpro 20 mg\nSECTION: Overview\n\nMaxpro 20 mg\nPatient Information Leaflet — Esomeprazole 20 mg. Read this leaflet carefully\nbefore you start taking this medicine.\nBrand name Maxpro 20 mg\nActive ingredient Esomeprazole 20 mg\nForm Enteric-coated (delayed-release) tablet\nManufacturer Renata PLC, Bangladesh\nPack & price Unit price ৳7.00; strip of 14: ৳98.00;\npack of 10 × 14: ৳980.00'}, {'id': 'maxpro_20mg_esomeprazole_leaflet.pdf::1. What Maxpro is and what it is used for::0', 'source': 'maxpro_20mg_esomeprazole_leaflet.pdf', 'medicine': 'Maxpro 20 mg', 'section':

# Meta data adding to chunks + convert chunks to langchain Documents

In [11]:
from langchain_core.documents import Document

documents = []

for chunk in chunks:
    document = Document(
        page_content=chunk["text"],
        metadata={
            "source": chunk["source"],
            "medicine": chunk["medicine"],
            "section": chunk["section"],
        }
    )

    documents.append(document)

print(f"Created {len(documents)} LangChain documents.")

Created 35 LangChain documents.


In [22]:
print(documents)
# print(f"\n ------fist document full------\n")
# print(documents[0])

print(f"\n------fist document page_content------\n")
print(documents[0].page_content)
print(f"\n\n------fist document metadata------\n")
print(documents[0].metadata)

[Document(metadata={'source': 'maxpro_20mg_esomeprazole_leaflet.pdf', 'medicine': 'Maxpro 20 mg', 'section': 'Overview'}, page_content='MEDICINE: Maxpro 20 mg\nSECTION: Overview\n\nMaxpro 20 mg\nPatient Information Leaflet — Esomeprazole 20 mg. Read this leaflet carefully\nbefore you start taking this medicine.\nBrand name Maxpro 20 mg\nActive ingredient Esomeprazole 20 mg\nForm Enteric-coated (delayed-release) tablet\nManufacturer Renata PLC, Bangladesh\nPack & price Unit price ৳7.00; strip of 14: ৳98.00;\npack of 10 × 14: ৳980.00'), Document(metadata={'source': 'maxpro_20mg_esomeprazole_leaflet.pdf', 'medicine': 'Maxpro 20 mg', 'section': '1. What Maxpro is and what it is used for'}, page_content='MEDICINE: Maxpro 20 mg\nSECTION: 1. What Maxpro is and what it is used for\n\nMaxpro contains esomeprazole, a proton pump inhibitor that reduces the\namount of acid produced in the stomach. It is used for:\nchronic heartburn and the symptoms of gastro-oesophageal reflux disease\n(GERD)\nhea

# Embedding 

In [24]:
%pip install "langchain-huggingface==1.2.2" "chromadb" "langchain-chroma==1.1.0" "sentence-transformers"

Note: you may need to restart the kernel to use updated packages.


In [25]:
from langchain_huggingface import HuggingFaceEmbeddings


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

/opt/miniconda3/envs/experiment/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7071.75it/s]


Embedding model loaded.


In [26]:
# vector = embeddings.embed_query("What are the side effects of Doxicap?")
# print(f"Embedding dimension: {len(vector)}")
# print(f"vector===>{vector}")

Embedding dimension: 384
vector===>[0.03222798556089401, -0.014210501685738564, 0.027544822543859482, 0.056318413466215134, 0.014251893386244774, -0.08549972623586655, 0.03313977271318436, 0.09699089825153351, 0.02463667094707489, -0.0432620607316494, -0.01730981655418873, -0.019672883674502373, 0.04641185700893402, 0.01848282478749752, -0.0731816217303276, -0.001611987128853798, 0.027048418298363686, 0.009548629634082317, 0.032372187823057175, 0.013004866428673267, 0.017310457304120064, -0.014783554710447788, 0.02423233352601528, 0.032788075506687164, -0.03356051445007324, 0.0630020722746849, -0.037231434136629105, 0.03783898055553436, 0.0278154406696558, -0.020057950168848038, -0.06215548887848854, 0.03863777965307236, 0.00641954131424427, -0.026933982968330383, -0.0347043052315712, -0.03688817843794823, 0.06461896747350693, 0.05489341542124748, -0.0016893342835828662, -0.02204125002026558, 0.024095090106129646, -0.04064198210835457, 0.019877946004271507, 0.0013710551429539919, -0.00

#  Store documents in ChromaDB

In [27]:
from langchain_chroma import Chroma


vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="./chroma_db",
    collection_name="medicine_leaflets",
)

print(f"Stored {len(documents)} documents in ChromaDB.")

Stored 35 documents in ChromaDB.


# Hybrid Search (Semantic + BM25 Keyword )

### Test semantic search

In [34]:
# query = "What are the possible side effects of Doxicap?"

# results = vectorstore.similarity_search_with_score(query, k=3)
# print(results)

# for document, score in results:
#     print("=" * 80)
#     print("DISTANCE:", score)
#     print("PAGE_CONTENT",document.page_content)
#     print("SOURCE:", document.metadata["source"])
#     print("MEDICINE:", document.metadata["medicine"])
#     print("SECTION:", document.metadata["section"])
#     print()

[(Document(id='503a1269-f5a8-4538-af1c-ba501d036bb9', metadata={'section': '4. Possible side effects', 'medicine': 'Doxicap 100 mg', 'source': 'doxicap_100mg_doxycycline_leaflet.pdf'}, page_content='MEDICINE: Doxicap 100 mg\nSECTION: 4. Possible side effects\n\nNausea, vomiting, diarrhoea, skin rashes, haemolytic anaemia, and eosinophilia\nmay occur. Contact your doctor if side effects are severe or persistent.'), 0.35558030009269714), (Document(id='9100b2ba-4356-4cff-8309-168a36ac7db4', metadata={'medicine': 'Doxicap 100 mg', 'source': 'doxicap_100mg_doxycycline_leaflet.pdf', 'section': '2. Before you take Doxicap'}, page_content='MEDICINE: Doxicap 100 mg\nSECTION: 2. Before you take Doxicap\n\nDo not take if you: are hypersensitive to any tetracycline; are a child under\n8 years of age; are pregnant; or are breast-feeding.\nUse during tooth development (from the last half of pregnancy through 8 years\nof age) may cause permanent discolouration of the teeth.'), 0.8082212805747986), (D

### Test BM25 keyword search

In [35]:
%pip install rank-bm25
%pip install "langchain-community"

  Using cached rank_bm25-0.2.2-py3-none-any.whl.metadata (3.2 kB)
Using cached rank_bm25-0.2.2-py3-none-any.whl (8.6 kB)
Note: you may need to restart the kernel to use updated packages.


## Create BM25 retriever with and without langchain

### without langchain create bm25 index

In [41]:
from rank_bm25 import BM25Okapi

# Get text from each document
texts = []

for document in documents:
    texts.append(document.page_content)

# Convert text into words
tokenized_texts = []

for text in texts:
    words = text.lower().split()
    tokenized_texts.append(words)

# Create BM25 index
bm25 = BM25Okapi(tokenized_texts)
print(bm25)

print(f"BM25 index created for {len(texts)} documents.")

BM25 index created for 35 documents.


##### Test

In [50]:
# query = "What are the possible side effects of Doxicap?"
query = "I am 5 years old. Can I take Doxicap?"

query_tokens = query.lower().split()

results = bm25.get_top_n(
    query_tokens,
    documents,
    n=5
)

for document in results:
    print("=" * 80)
    print(document.page_content)
    print("SOURCE:", document.metadata["source"])
    print("MEDICINE:", document.metadata["medicine"])
    print("SECTION:", document.metadata["section"])

    print()


MEDICINE: Rolip 10 mg
SECTION: 3. How to take Rolip

Can be taken with or without food, at any time of day.
Patient Dose
Usual range (adults) 5–40 mg once daily
Very high cholesterol / high risk up to 40 mg once daily
Homozygous familial hypercholesterolaemiastart 20 mg once daily
Asian patients consider starting at 5 mg once
daily
Severe renal impairment (not on dialysis)
Patient Dose
start at 5 mg, not exceeding
10 mg daily
If you also take cyclosporine, limit to 5 mg daily; with certain protease-inhibitor
combinations, limit to 10 mg daily.
SOURCE: rolip_10mg_rosuvastatin_leaflet.pdf
MEDICINE: Rolip 10 mg
SECTION: 3. How to take Rolip

MEDICINE: Doxicap 100 mg
SECTION: 2. Before you take Doxicap

Do not take if you: are hypersensitive to any tetracycline; are a child under
8 years of age; are pregnant; or are breast-feeding.
Use during tooth development (from the last half of pregnancy through 8 years
of age) may cause permanent discolouration of the teeth.
SOURCE: doxicap_100mg_dox

### With langchain create bm25 retriever

In [51]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 5

print(bm25_retriever)
print(f"BM25 retriever created for {len(documents)} documents.")

vectorizer=<rank_bm25.BM25Okapi object at 0x13e0dd250> k=5
BM25 retriever created for 35 documents.


In [52]:
# query = "What are the possible side effects of Doxicap?"
query = "I am 5 years old. Can I take Doxicap?"


results = bm25_retriever.invoke(query)

for document in results:
    print("=" * 80)
    print(document.page_content)
    print("SOURCE:", document.metadata["source"])
    print("MEDICINE:", document.metadata["medicine"])
    print("SECTION:", document.metadata["section"])

    print()


MEDICINE: Rolip 10 mg
SECTION: 3. How to take Rolip

Can be taken with or without food, at any time of day.
Patient Dose
Usual range (adults) 5–40 mg once daily
Very high cholesterol / high risk up to 40 mg once daily
Homozygous familial hypercholesterolaemiastart 20 mg once daily
Asian patients consider starting at 5 mg once
daily
Severe renal impairment (not on dialysis)
Patient Dose
start at 5 mg, not exceeding
10 mg daily
If you also take cyclosporine, limit to 5 mg daily; with certain protease-inhibitor
combinations, limit to 10 mg daily.
SOURCE: rolip_10mg_rosuvastatin_leaflet.pdf
MEDICINE: Rolip 10 mg
SECTION: 3. How to take Rolip

MEDICINE: Doxicap 100 mg
SECTION: 2. Before you take Doxicap

Do not take if you: are hypersensitive to any tetracycline; are a child under
8 years of age; are pregnant; or are breast-feeding.
Use during tooth development (from the last half of pregnancy through 8 years
of age) may cause permanent discolouration of the teeth.
SOURCE: doxicap_100mg_dox

# Compare semantic vs keyword search

#### Raw obstay test korlam

In [70]:
from langchain_community.retrievers import BM25Retriever


query = "I am 5 years old. Can I take Doxicap?"

# Semantic Search
semantic_results = vectorstore.similarity_search_with_score(query, k=5)
# or
# semantic_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
# semantic_results = semantic_retriever.invoke(query)

print("SEMANTIC SEARCH")
print("=" * 80)
for doc, distance in semantic_results:
    print( f"Distance: {round(distance, 4)} | {doc.metadata["medicine"]} | {doc.metadata["section"]} ")




# BM25 Search
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 5
bm25_results = bm25_retriever.invoke(query)

print("\nBM25 SEARCH")
print("=" * 80)

for doc in bm25_results:
    print(doc.metadata["medicine"], "|", doc.metadata["section"])

SEMANTIC SEARCH
Distance: 0.5009 | Doxicap 100 mg | 2. Before you take Doxicap 
Distance: 0.5926 | Doxicap 100 mg | 5. Use in pregnancy and breast-feeding 
Distance: 0.63 | Doxicap 100 mg | 4. Possible side effects 
Distance: 0.6359 | Doxicap 100 mg | 3. How to take Doxicap 
Distance: 0.7272 | Doxicap 100 mg | 1. What Doxicap is and what it is used for 

BM25 SEARCH
Rolip 10 mg | 3. How to take Rolip
Doxicap 100 mg | 2. Before you take Doxicap
Rolac 10 mg | 2. Before you take Rolac
Fenadin 120 mg | Overview
Fenadin 120 mg | 1. What Fenadin is and what it is used for


#### Function banaye test korlam

In [71]:
from langchain_community.retrievers import BM25Retriever


query = "I am 5 years old. Can I take Doxicap?"


# ----------- Semantic Search ----------------

semantic_results = vectorstore.similarity_search_with_score(query, k=5)

print("SEMANTIC SEARCH")
print("=" * 80)
for doc, distance in semantic_results:
    print( f"Distance: {round(distance, 4)} | {doc.metadata["medicine"]} | {doc.metadata["section"]} ")




# ----------- BM25 Search ----------------

def create_bm25_retriever(documents, k=5):
    retriever = BM25Retriever.from_documents(documents)
    retriever.k = k
    return retriever

bm25_retriever = create_bm25_retriever(documents, k=5)
bm25_results = bm25_retriever.invoke(query)

print("\nBM25 SEARCH")
print("=" * 80)

for doc in bm25_results:
    print(f"{doc.metadata["medicine"]} | {doc.metadata["section"]}")

SEMANTIC SEARCH
Distance: 0.5009 | Doxicap 100 mg | 2. Before you take Doxicap 
Distance: 0.5926 | Doxicap 100 mg | 5. Use in pregnancy and breast-feeding 
Distance: 0.63 | Doxicap 100 mg | 4. Possible side effects 
Distance: 0.6359 | Doxicap 100 mg | 3. How to take Doxicap 
Distance: 0.7272 | Doxicap 100 mg | 1. What Doxicap is and what it is used for 

BM25 SEARCH
Rolip 10 mg | 3. How to take Rolip
Doxicap 100 mg | 2. Before you take Doxicap
Rolac 10 mg | 2. Before you take Rolac
Fenadin 120 mg | Overview
Fenadin 120 mg | 1. What Fenadin is and what it is used for


# Hybrid Retriver create by combining (Keyword search BM25 + Semantic Search)

- Get top results from semantic search. (5 chunks)
- Get top results from BM25. (5 chunks)
- Merge them.(5 to 10 chunks)
- Remove duplicates.(lets say 7 chunks from those 10 chunks)
- Return top k.

In [85]:




from langchain_community.retrievers import BM25Retriever





def hybrid_search(query, k=5):
    # Semantic search
    semantic_results = vectorstore.similarity_search_with_score(query, k=k)

    # BM25 search
    bm25_retriever = BM25Retriever.from_documents(documents)
    bm25_retriever.k = k
    bm25_results = bm25_retriever.invoke(query)

    results = []
    seen = set()

    # Add semantic results
    for document, score in semantic_results:
        doc_id = (
            document.metadata["source"]
            + "::"
            + document.metadata["section"]
            + "::"
            + document.page_content
        )

        if doc_id not in seen:
            results.append(document)
            seen.add(doc_id)

    # Add BM25 results
    for document in bm25_results:
        doc_id = (
            document.metadata["source"]
            + "::"
            + document.metadata["section"]
            + "::"
            + document.page_content
        )

        if doc_id not in seen:
            results.append(document)
            seen.add(doc_id)

    return results

In [89]:
# query = "I am 5 years old. Can I take Doxicap?"

# candidates = hybrid_search(query, k=5)
# print(candidates)
# print(f"total number of unique candidates chunks ==> {len(candidates)}" )

# # for doc in candidates:
# #     print("=" * 80)
# #     print(doc.metadata["medicine"], "|", doc.metadata["section"])
# #     print(doc.page_content)

[Document(id='9100b2ba-4356-4cff-8309-168a36ac7db4', metadata={'medicine': 'Doxicap 100 mg', 'source': 'doxicap_100mg_doxycycline_leaflet.pdf', 'section': '2. Before you take Doxicap'}, page_content='MEDICINE: Doxicap 100 mg\nSECTION: 2. Before you take Doxicap\n\nDo not take if you: are hypersensitive to any tetracycline; are a child under\n8 years of age; are pregnant; or are breast-feeding.\nUse during tooth development (from the last half of pregnancy through 8 years\nof age) may cause permanent discolouration of the teeth.'), Document(id='deb85253-ce21-492c-b70e-c4ec31098bea', metadata={'section': '5. Use in pregnancy and breast-feeding', 'source': 'doxicap_100mg_doxycycline_leaflet.pdf', 'medicine': 'Doxicap 100 mg'}, page_content='MEDICINE: Doxicap 100 mg\nSECTION: 5. Use in pregnancy and breast-feeding\n\nAvoid during pregnancy due to the risk of staining of the baby’s teeth and\neffects on bone growth. The drug enters breast milk, so nursing mothers should\nnot breast-feed whi

# Add A Reranker 

In [ ]:
# %pip install sentence-transformers
# %pip install langchain-community sentence-transformers

In [96]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

reranker_model = HuggingFaceCrossEncoder( model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 18305.84it/s]


In [ ]:
# from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# reranker_model = HuggingFaceCrossEncoder( model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank_documents(query, documents,k=5):

    pairs = []
    for document in documents:
        pair = [query, document.page_content]
        pairs.append(pair)

    scores = reranker_model.score(pairs)
    
    scored_documents = []
    
    print(f"Total Candidate Douments/chunks = {len(documents)}")
    print("=" * 50)
    
    # for document, score in zip(documents, scores):
    #     # print("Document:", document.page_content)
    #     print("Cross-Encoder score:", score)
    #     print("-" * 50)
    #     scored_documents.append([document, score])

    for i, (document, score) in enumerate(zip(documents, scores), start=1):
        print(f"Chunk {i}")
        print(f"Cross-Encoder Score: {score}")
        # print("-" * 50)
        scored_documents.append([document, score])


    scored_documents.sort(key=lambda item: item[1], reverse=True)

    results = []

    print(f"After giving Score by Reranker model Top {k} Chunks are: ")
    for document, score in scored_documents[:k]:
        # print("Document:", document.page_content)
        print("Final Score:", score)
        results.append(document)

    return results


In [107]:
# query = "I am 5 years old. Can I take Doxicap?"

# # Get hybrid candidates
# candidates = hybrid_search(query, k=5)
# print(candidates)
# # print("=" * 80)
# # print("Total  hybrid candidates chunks:", len(candidates))

# # Rerank candidates

# final_reranked_results = rerank_documents(query,candidates,k=5)
# print(final_reranked_results)


# # # Show final results
# # print("\nFINAL RERANKED RESULTS")
# # print("=" * 80)

# # for document in final_reranked_results:
# #     print(document.metadata["medicine"],"|",document.metadata["section"])
# #     print(document.page_content)
# #     print("-" * 80)

[Document(id='9100b2ba-4356-4cff-8309-168a36ac7db4', metadata={'source': 'doxicap_100mg_doxycycline_leaflet.pdf', 'section': '2. Before you take Doxicap', 'medicine': 'Doxicap 100 mg'}, page_content='MEDICINE: Doxicap 100 mg\nSECTION: 2. Before you take Doxicap\n\nDo not take if you: are hypersensitive to any tetracycline; are a child under\n8 years of age; are pregnant; or are breast-feeding.\nUse during tooth development (from the last half of pregnancy through 8 years\nof age) may cause permanent discolouration of the teeth.'), Document(id='deb85253-ce21-492c-b70e-c4ec31098bea', metadata={'section': '5. Use in pregnancy and breast-feeding', 'source': 'doxicap_100mg_doxycycline_leaflet.pdf', 'medicine': 'Doxicap 100 mg'}, page_content='MEDICINE: Doxicap 100 mg\nSECTION: 5. Use in pregnancy and breast-feeding\n\nAvoid during pregnancy due to the risk of staining of the baby’s teeth and\neffects on bone growth. The drug enters breast milk, so nursing mothers should\nnot breast-feed whi

# Create Context using those reranked chunks for Ollama

In [111]:
def create_context(documents):

    context = ""

    for i, document in enumerate(documents):
        context += f"\nDocument {i + 1}:\n"
        context += document.page_content
        context += "\n"

    return context


In [113]:
# query = "I am 5 years old. Can I take Doxicap?"

# # Get hybrid candidates
# candidates = hybrid_search(query, k=5)
# print(candidates)

# final_reranked_results = rerank_documents(query,candidates,k=5)
# print(final_reranked_results)

# context = create_context(final_reranked_results)
# print(f"final context to ollama ====> {context}")


[Document(id='9100b2ba-4356-4cff-8309-168a36ac7db4', metadata={'section': '2. Before you take Doxicap', 'source': 'doxicap_100mg_doxycycline_leaflet.pdf', 'medicine': 'Doxicap 100 mg'}, page_content='MEDICINE: Doxicap 100 mg\nSECTION: 2. Before you take Doxicap\n\nDo not take if you: are hypersensitive to any tetracycline; are a child under\n8 years of age; are pregnant; or are breast-feeding.\nUse during tooth development (from the last half of pregnancy through 8 years\nof age) may cause permanent discolouration of the teeth.'), Document(id='deb85253-ce21-492c-b70e-c4ec31098bea', metadata={'source': 'doxicap_100mg_doxycycline_leaflet.pdf', 'medicine': 'Doxicap 100 mg', 'section': '5. Use in pregnancy and breast-feeding'}, page_content='MEDICINE: Doxicap 100 mg\nSECTION: 5. Use in pregnancy and breast-feeding\n\nAvoid during pregnancy due to the risk of staining of the baby’s teeth and\neffects on bone growth. The drug enters breast milk, so nursing mothers should\nnot breast-feed whi

# Ollama Calling

In [114]:
%pip install  langchain langchain-core langchain-community langchain-ollama

  Using cached langchain-1.3.15-py3-none-any.whl.metadata (6.1 kB)
  Using cached langchain_ollama-1.1.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached langgraph-1.2.11-py3-none-any.whl.metadata (4.9 kB)
  Using cached langgraph_checkpoint-4.2.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.4.2-py3-none-any.whl.metadata (3.6 kB)
  Using cached ormsgpack-1.12.2-cp312-cp312-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (3.2 kB)
  Using cached websockets-15.0.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached ollama-0.6.2-py3-none-any.whl.metadata (5.8 kB)
Using cached langchain-1.3.15-py3-none-any.whl (147 kB)
Using cached langgraph-1.2.11-py3-none-any.whl (248 kB)
Using cached langgraph_checkpoint-4.2.0-py3-none-any.whl (56 kB)
Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl (41 kB)
Using cached langgraph_sdk-0.4.2-py3-none-any.whl (160 

In [116]:
from langchain_ollama import ChatOllama
llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)



def generate_answer(query, context):


    prompt = f"""
    Answer the question using only the provided context.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    response = llm.invoke(prompt)

    return response.content



In [123]:
query = "What are the side effects of doxicap?"


# Get hybrid candidates
candidates = hybrid_search(query, k=5)
# print(candidates)

final_reranked_results = rerank_documents(query,candidates,k=5)
# print(final_reranked_results)

context = create_context(final_reranked_results)
# print(f"final context to ollama ====> {context}")



answer = generate_answer(query,context)
print("=" * 80)
print(f"Ollama final answer=====> {answer}")


Total Candidate Douments/chunks = 8
Chunk 1
Cross-Encoder Score: 8.528883934020996
Chunk 2
Cross-Encoder Score: 3.193821430206299
Chunk 3
Cross-Encoder Score: -1.153989315032959
Chunk 4
Cross-Encoder Score: 3.3309950828552246
Chunk 5
Cross-Encoder Score: 3.049697160720825
Chunk 6
Cross-Encoder Score: -1.3813213109970093
Chunk 7
Cross-Encoder Score: -0.7860954999923706
Chunk 8
Cross-Encoder Score: -1.4861767292022705
After giving Score by Reranker model Top 5 Chunks are: 
Final Score: 8.528884
Final Score: 3.330995
Final Score: 3.1938214
Final Score: 3.0496972
Final Score: -0.7860955
Ollama final answer=====> According to Document 1, the possible side effects of Doxicap are:

Nausea, vomiting, diarrhoea, skin rashes, haemolytic anaemia, and eosinophilia.
